GOAL: CIFAR10 image generation using Logistic noising.

In [ ]:
import torch
import numpy as np
import torchvision
import torch.nn as nn
from torchvision.datasets import CIFAR10 as CIFAR10

In [ ]:
transform = torchvision.transforms.Compose([
    torchvision.transforms.Grayscale(num_output_channels=1), # Keep as 3 channels for now
    torchvision.transforms.ToTensor(),
    # Removed normalization to keep data in [0, 1] range for consistency with Noising function and loss
    # torchvision.transforms.Normalize((0.5,), (0.5,))
])

# Load the full CIFAR-10 training data
full_train_data = CIFAR10(root="./data", train=True, transform=transform, download=True)

# Filter the dataset to include only images with a specific label
target_label = 3 # CAT IMAGES
indices = [i for i, (image, label) in enumerate(full_train_data) if label == target_label]
train_data = torch.utils.data.Subset(full_train_data, indices)

100%|██████████| 170M/170M [00:10<00:00, 16.1MB/s] 


In [ ]:
def Noising(data_tensor, num_steps, theta = 4):
    for i in range(num_steps):
        data_tensor = theta * data_tensor * (1 - data_tensor)
    noisy_image = data_tensor
    return noisy_image, num_steps

In [ ]:
import torch.nn as nn
import torch
import torch

class rnn(nn.Module):
    def __init__(self, input_dim, state_dim):
        super(rnn, self).__init__()
        self.state_dim = state_dim
        self.input_dim = input_dim
        # Using batch_first=True so input shape is (batch, seq_len, input_dim)
        self.rnn = nn.RNN(input_size=input_dim, hidden_size=state_dim, batch_first=True)
        # map final hidden to flattened image
        self.fc = nn.Linear(state_dim, input_dim)

    def forward(self, x):
        # x: (batch_size, sequence_length, input_dim)
        rnn_out, _ = self.rnn(x)  # rnn_out: (batch, seq_len, state_dim)
        last_step_output = rnn_out[:, -1, :]  # (batch, state_dim)
        denoised_image = torch.sigmoid(self.fc(last_step_output))  # (batch, input_dim) in [0,1]
        return denoised_image

In [ ]:
from torch.utils.data import DataLoader
import torch.optim as optim
import torch
import random
import torch.nn.functional as F

def train(model, train_data, num_epochs, max_noising_steps=5, batch_size=64, device="cpu"): # Renamed noising_steps to max_noising_steps
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.1)
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)

    lambda_l1 = 0.0001

    model.to(device)

    for epoch in range(num_epochs):
        total_loss = 0.0

        for batch_idx, (images, labels) in enumerate(train_loader):
            images = images.to(device)

            # Generate a sequence of noisy images with decreasing noise levels
            noisy_sequence = []
            original_images_flat = images.view(images.size(0), -1) # Flatten original images once

            for num_steps in range(0, max_noising_steps + 1, -1):
                # Ensure images passed to Noising are on the correct device
                noisy_image, num_steps = Noising(images.clone().to(device), num_steps=num_steps, theta=4.0) # Apply noising
                # Ensure noisy_image is on the correct device before flattening
                noisy_sequence.append(noisy_image.to(device).view(images.size(0), -1)) # Flatten and add to sequence


            # Stack the noisy images along the sequence dimension (batch_size, sequence_length, input_dim)
            noisy_sequence_tensor = torch.stack(noisy_sequence, dim=1).to(device)


            # Feed the noisy sequence to the RNN model
            denoised_images_flat = model(noisy_sequence_tensor)


            # Calculate the loss (e.g., MSE between denoised image and original image)
            # Compare the denoised output to the original, clean image
            loss = F.mse_loss(denoised_images_flat, original_images_flat)


            l1_reg = torch.tensor(0.).to(device)
            for param in model.parameters():
                l1_reg += torch.norm(param, 1)

            loss = loss + lambda_l1 * l1_reg


            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() # Use item() to get the scalar loss value

        scheduler.step()

        avg_loss = total_loss / len(train_loader.dataset) # Calculate average loss per sample
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.8f}")


# Instantiate the RNN model
input_dim = 32 * 32  # CIFAR-10 image dimensions
state_dim = 128 # RNN state dimension
hidden_dim = 128 # Hidden dimension for linear layers in RNN
# model = rnn(input_dim, state_dim, hidden_dim) # Model instantiation moved inside the cell

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device) # Moved inside the train function

# Use train_data which is the CIFAR-10 data filtered for cats.
train(model=rnn(input_dim, state_dim), train_data=train_data, num_epochs=20, max_noising_steps=100, device=device) # Instantiate and pass model directly with 2 arguments